# Preprocessing — Transformaciones de datos

**Objetivo:** Transformar el dataset enriquecido en datos numéricos, sin nulos, listos para análisis.

En este notebook:
1. Cargamos el dataset de feature engineering
2. Imputamos nulos con estrategia definitiva
3. Encoding de categóricas
4. Guardamos datos transformados

**NOTA:** El train/test split y el scaling van en `08_modeling.ipynb`, después de los test estadísticos y feature selection.

**Regla:** Este notebook transforma TODO el dataset. Los test estadísticos necesitan todos los datos.

---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print('Setup listo.')

---
## 2. Carga de datos

In [ ]:
df = pd.read_csv('../data/processed/application_train_features.csv')

print(f'Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'Target:')
print(df['TARGET'].value_counts())

---
## 3. Identificar tipos de columnas

In [ ]:
TARGET_COL = 'TARGET'
ID_COL = 'SK_ID_CURR'

num_cols = df.drop(columns=[ID_COL, TARGET_COL]).select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.drop(columns=[ID_COL, TARGET_COL]).select_dtypes(exclude=[np.number]).columns.tolist()

print(f'Numéricas: {len(num_cols)}')
print(f'Categóricas: {len(cat_cols)}')
print()
for c in cat_cols:
    print(f'  {c:40s} → {df[c].nunique()} categorías')

---
## 4. Imputación de nulos

Estrategia:
- **Numéricas:** mediana (robusta a outliers)
- **Categóricas:** moda (valor más frecuente)

In [ ]:
# Nulos antes de imputar
nulls_before = df.isnull().sum()
nulls_before = nulls_before[nulls_before > 0].sort_values(ascending=False)

print(f'Columnas con nulos: {len(nulls_before)} de {df.shape[1]}')
print(f'Total nulos: {nulls_before.sum():,}')
print()
print('Top 10:')
for col, n in nulls_before.head(10).items():
    print(f'  {col:40s} → {n:>8,} ({n/len(df)*100:.1f}%)')

In [ ]:
# Imputar numéricas con mediana
imputation_medians = {}
for col in num_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        imputation_medians[col] = median_val
        df[col] = df[col].fillna(median_val)

# Imputar categóricas con moda
imputation_modes = {}
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        imputation_modes[col] = mode_val
        df[col] = df[col].fillna(mode_val)

# Verificar
nulls_after = df.isnull().sum().sum()
print(f'Nulos después de imputar: {nulls_after}')
print(f'Medianas guardadas: {len(imputation_medians)}')
print(f'Modas guardadas: {len(imputation_modes)}')

---
## 5. Encoding de categóricas

Estrategia:
- **Binarias** (2 categorías): Label Encoding (0/1)
- **Multi-categoría** (>2 categorías): One-Hot Encoding

In [ ]:
# Separar binarias de multi-categoría
binary_cols = [c for c in cat_cols if df[c].nunique() == 2]
multi_cols = [c for c in cat_cols if df[c].nunique() > 2]

print(f'Binarias (label encoding): {len(binary_cols)}')
for c in binary_cols:
    print(f'  {c:40s} → {df[c].unique()}')
print()
print(f'Multi-categoría (one-hot): {len(multi_cols)}')
for c in multi_cols:
    print(f'  {c:40s} → {df[c].nunique()} categorías')

In [ ]:
# Label Encoding para binarias
label_encoders = {}
for col in binary_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

print('Label encoding completado.')

In [ ]:
# One-Hot Encoding para multi-categoría
df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

print(f'One-hot encoding completado.')
print(f'Columnas después de one-hot: {df.shape[1]}')

In [ ]:
# Verificar que no quedan categóricas
remaining_cat = df.drop(columns=[ID_COL, TARGET_COL]).select_dtypes(exclude=[np.number]).columns.tolist()
print(f'Categóricas restantes: {len(remaining_cat)}')
if remaining_cat:
    print(remaining_cat)

---
## 6. Resumen final

In [ ]:
print('RESUMEN DEL PREPROCESSING')
print('=' * 50)
print(f'Features finales:  {df.shape[1] - 2}')
print(f'Muestras:          {df.shape[0]:,}')
print(f'Nulos restantes:   {df.isnull().sum().sum()}')
print()
print('Transformaciones aplicadas:')
print(f'  - Imputación: mediana ({len(imputation_medians)} cols) / moda ({len(imputation_modes)} cols)')
print(f'  - Label encoding: {len(binary_cols)} columnas binarias')
print(f'  - One-hot encoding: {len(multi_cols)} columnas multi-categoría')
print()
print('PRÓXIMOS PASOS:')
print('  1. 06_statistical_tests.ipynb — Testeos de hipótesis')
print('  2. 07_feature_selection.ipynb — Selección de features')
print('  3. 08_modeling.ipynb — Train/test split + Scaling + Entrenamiento')

---
## 7. Guardar datos transformados

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/application_train_preprocessed.csv', index=False)

print(f'Guardado: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'Ruta: data/processed/application_train_preprocessed.csv')